In [71]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [72]:
df = pd.read_csv("cleaned_matches.csv")

In [73]:
df = df[['team1','team2','venue','toss_winner','toss_decision','winner']]

In [74]:
# Feature 1: Did team1 win toss?
df['team1_toss_win'] = (df['team1'] == df['toss_winner']).astype(int)

# Feature 2: Convert toss decision to numeric
df['toss_decision'] = df['toss_decision'].map({'bat': 1, 'field': 0})

# Check result
df.head(5)

,team1,team2,venue,toss_winner,toss_decision,winner,team1_toss_win
0,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Royal Challengers Bengaluru,0,Kolkata Knight Riders,1
1,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,1,Chennai Super Kings,0
2,Delhi Daredevils,Rajasthan Royals,Feroz Shah Kotla,Rajasthan Royals,1,Delhi Daredevils,0
3,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai Indians,1,Royal Challengers Bengaluru,1
4,Kolkata Knight Riders,Deccan Chargers,Eden Gardens,Deccan Chargers,1,Kolkata Knight Riders,0


In [75]:
# Apply One-Hot Encoding on categorical columns
df = pd.get_dummies(df, columns=['team1', 'team2', 'toss_winner', 'venue'])

In [76]:
# Initialize encoder
le = LabelEncoder()

# Convert winner column into numeric values
df['winner'] = le.fit_transform(df['winner'])

In [77]:
# Split into X and y
X = df.drop('winner', axis=1)
y = df['winner']

In [86]:
# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [87]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [88]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

RandomForestClassifier()

In [89]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

C:\Users\Parth\coding tools\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:59:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [90]:
# Predictions
y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb.predict(X_test)

In [91]:
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("Random Forest:", accuracy_score(y_test, y_pred_rf))
print("XGBoost:", accuracy_score(y_test, y_pred_xgb))

Logistic Regression: 0.5321100917431193
Random Forest: 0.5137614678899083
XGBoost: 0.5168195718654435


In [92]:
print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))

XGBoost Report:
               precision    recall  f1-score   support

           0       0.56      0.56      0.56        45
           1       0.56      0.83      0.67         6
           2       0.33      0.20      0.25        15
           3       0.56      0.45      0.50        20
           4       0.67      0.57      0.62         7
           5       0.78      0.58      0.67        12
           6       0.00      0.00      0.00         2
           7       0.55      0.46      0.50        39
           8       0.40      0.67      0.50         3
           9       0.45      0.51      0.48        37
          10       0.33      0.25      0.29         4
          11       0.38      0.32      0.34        38
          12       0.58      0.61      0.59        31
          13       0.40      0.25      0.31         8
          14       0.56      0.72      0.63        39
          15       0.50      0.71      0.59        21

    accuracy                           0.52       327
   macro 

C:\Users\Parth\coding tools\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Parth\coding tools\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Parth\coding tools\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [93]:
#overfitting and underfitting
print("Train Accuracy:", xgb.score(X_train, y_train))
print("Test Accuracy:", xgb.score(X_test, y_test))

Train Accuracy: 0.9134993446920052
Test Accuracy: 0.5168195718654435
